# Load model and Train

In [3]:
"""
train_rollout.py

Rollout (multi-step) training for PhysicsNN: instead of training on
independent (y, ydot) points, this trains on short contiguous sequences
per trajectory -- predicting ydot, integrating forward one step at a
time using the model's OWN predictions, and comparing the resulting
predicted states against the true states. This directly penalizes
error accumulation over time, unlike pointwise training.

Keep the existing pointwise training script as-is for comparison --
this is a separate, slower, harder training regime.
"""

import sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from models.PhysicsNN import PhysicsNN

DATA_PATH = "../data/raw/random_state_data.npz"
VAL_FRACTION = 0.2
N_STEPS = 1                 # rollout length (number of integration steps per sequence)
MAX_SEQS_PER_TRAJ = 5        # how many random windows to sample per trajectory
N_EPOCHS = 20
LR = 1e-4                    # smaller than pointwise training -- rollout is less stable
GRAD_CLIP = 1.0
SEED = 0

# --- Load data ---
data = np.load(DATA_PATH)
y_all, ydot_all, t_all, traj_id_all = data["Y"], data["Ydot"], data["t"], data["traj_id"]

print(y_all.shape, ydot_all.shape, t_all.shape)

rng = np.random.default_rng(SEED)

unique_trajs = np.unique(traj_id_all)
rng.shuffle(unique_trajs)

n_val_trajs = int(len(unique_trajs) * VAL_FRACTION)
val_trajs = unique_trajs[:n_val_trajs]
train_trajs = unique_trajs[n_val_trajs:]

print(f"train trajectories: {len(train_trajs)}, val trajectories: {len(val_trajs)}")


# --- Build contiguous windows (sequences) per trajectory ---
def build_sequences(traj_ids, n_steps, rng, max_seqs_per_traj):
    sequences = []
    for tid in traj_ids:
        idx = np.where(traj_id_all == tid)[0]
        if len(idx) < n_steps + 1:
            continue
        n_possible = len(idx) - n_steps
        n_pick = min(max_seqs_per_traj, n_possible)
        starts = rng.choice(n_possible, size=n_pick, replace=False)
        for s in starts:
            window = idx[s:s + n_steps + 1]
            sequences.append(window)
    return sequences


train_sequences = build_sequences(train_trajs, N_STEPS, rng, MAX_SEQS_PER_TRAJ)
val_sequences = build_sequences(val_trajs, N_STEPS, rng, MAX_SEQS_PER_TRAJ)

print(f"train sequences: {len(train_sequences)}, val sequences: {len(val_sequences)}")


# --- Normalization stats (computed from a flat sample of train points, same as pointwise script) ---
sample_idx = np.concatenate(train_sequences)[:200_000]
y_sample = y_all[sample_idx]

T_sample = y_sample[:, 0:1]
Y_sample = y_sample[:, 1:]
logY_sample = np.log(np.clip(Y_sample, 1e-30, None))

T_mean = torch.tensor(T_sample.mean(), dtype=torch.float32)
T_std = torch.tensor(T_sample.std(), dtype=torch.float32)
logY_mean = torch.tensor(logY_sample.mean(axis=0), dtype=torch.float32)
logY_std = torch.tensor(logY_sample.std(axis=0) + 1e-8, dtype=torch.float32)

print("T stats:", T_mean.item(), T_std.item())


def normalize_state(y_phys):
    """y_phys: (54,) tensor, physical units -> normalized input for the model"""
    T = y_phys[0:1]
    Y = y_phys[1:].clamp(min=1e-30)
    T_norm = (T - T_mean) / T_std
    logY_norm = (torch.log(Y) - logY_mean) / logY_std
    return torch.cat([T_norm, logY_norm])


def unnormalize_ydot(ydot_norm):
    """model output (signed-log space) -> physical units"""
    return torch.sign(ydot_norm) * torch.expm1(torch.abs(ydot_norm))


# --- Rollout loss for one sequence ---
def rollout_loss(model, window_idx):
    y_window = torch.tensor(y_all[window_idx], dtype=torch.float32)   # (n_steps+1, 54)
    t_window = t_all[window_idx]                                       # (n_steps+1,)

    y_pred = y_window[0]   # start from the TRUE initial state
    loss = 0.0

    for step in range(len(t_window) - 1):
        dt = float(t_window[step + 1] - t_window[step])

        y_norm = normalize_state(y_pred).unsqueeze(0)
        ydot_norm = model(y_norm).squeeze(0)
        ydot_phys = unnormalize_ydot(ydot_norm)

        y_pred = y_pred + dt * ydot_phys

        y_true_next = y_window[step + 1]
        loss = loss + torch.mean((y_pred - y_true_next) ** 2)

    return loss / (len(t_window) - 1)


# --- Model, optimizer ---
device = torch.device("cpu")  # keep on CPU -- this loop is inherently sequential, GPU won't help much here
model = PhysicsNN(dim=54, hidden=256).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_losses, val_losses = [], []

for epoch in range(N_EPOCHS):
    model.train()
    rng.shuffle(train_sequences)
    epoch_loss = 0.0

    for window_idx in train_sequences:
        optimizer.zero_grad()
        loss = rollout_loss(model, window_idx)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= len(train_sequences)
    train_losses.append(epoch_loss)

    model.eval()
    val_epoch_loss = 0.0
    with torch.no_grad():
        for window_idx in val_sequences:
            val_epoch_loss += rollout_loss(model, window_idx).item()
    val_epoch_loss /= len(val_sequences)
    val_losses.append(val_epoch_loss)

    print(f"epoch {epoch+1}/{N_EPOCHS}  train rollout loss {epoch_loss:.4e}  val rollout loss {val_epoch_loss:.4e}")

plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel(f"Rollout MSE loss (physical space, {N_STEPS} steps)")
plt.legend()
plt.show()


# --- Diagnostic: plot one example rollout vs true trajectory ---
example_window = val_sequences[0]
y_true_seq = y_all[example_window]
t_seq = t_all[example_window]

model.eval()
with torch.no_grad():
    y_pred = torch.tensor(y_true_seq[0], dtype=torch.float32)
    preds = [y_pred.numpy()]
    for step in range(len(t_seq) - 1):
        dt = float(t_seq[step + 1] - t_seq[step])
        y_norm = normalize_state(y_pred).unsqueeze(0)
        ydot_norm = model(y_norm).squeeze(0)
        ydot_phys = unnormalize_ydot(ydot_norm)
        y_pred = y_pred + dt * ydot_phys
        preds.append(y_pred.numpy())

preds = np.array(preds)

plt.plot(t_seq, y_true_seq[:, 0], 'o-', label="True T")
plt.plot(t_seq, preds[:, 0], 'x--', label="Predicted T (rollout)")
plt.xlabel("t")
plt.ylabel("Temperature (K)")
plt.legend()
plt.show()

(18336085, 54) (18336085, 54) (18336085,)
train trajectories: 8000, val trajectories: 2000


KeyboardInterrupt: 

In [4]:
N_STEPS = 1

train_sequences = build_sequences(train_trajs, N_STEPS, rng, MAX_SEQS_PER_TRAJ)
val_sequences = build_sequences(val_trajs, N_STEPS, rng, MAX_SEQS_PER_TRAJ)

print(f"train sequences: {len(train_sequences)}")

window_idx = train_sequences[0]

optimizer.zero_grad()
loss = rollout_loss(model, window_idx)
print("loss:", loss.item())

loss.backward()

total_grad_norm = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
print("total grad norm:", total_grad_norm)

for name, p in model.named_parameters():
    if p.grad is not None:
        print(name, p.grad.norm().item())

train sequences: 40000
loss: 0.0023399677593261003
total grad norm: 1.2370448707001071e-11
net.0.weight 4.456302135028706e-13
net.0.bias 1.1276742750308474e-13
net.2.weight 9.569954420932114e-13
net.2.bias 2.3507422137331035e-13
net.4.weight 1.3924697983211387e-12
net.4.bias 7.708231297177959e-13
net.6.weight 6.634814225803254e-13
net.6.bias 7.793207051909334e-12


In [7]:
t_window = t_all[window_idx]
print("dt:", t_window[1] - t_window[0])

dt: 5.795307367975632e-10
